# Final flagship notebook 01 — truth regimes

This notebook builds the flagship truth-based circulation regime labels for the paper window and writes them to a dedicated flagship output directory.

**Flagship defaults baked in**
- uses the patched bundle root detection
- writes to `outputs/flagship_final/regimes`
- keeps the main paper configuration at **4 regimes / 8 EOF components**
- keeps the current paper window, but exposes a single override cell so you can broaden it later if needed

In [1]:
from pathlib import Path
import sys
import json
from copy import deepcopy
import pandas as pd
from IPython.display import display, Markdown

def detect_bundle_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    for c in candidates:
        if (c / "src" / "flagship_predictability").exists() and (c / "examples" / "default_settings.py").exists():
            return c
    raise RuntimeError("Could not find bundle root with src/flagship_predictability and examples/default_settings.py")

BUNDLE_ROOT = detect_bundle_root()
if str(BUNDLE_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(BUNDLE_ROOT / "src"))
if str(BUNDLE_ROOT) not in sys.path:
    sys.path.insert(0, str(BUNDLE_ROOT))

FLAGSHIP_OUTPUT_ROOT = (BUNDLE_ROOT / "notebooks" / "outputs" / "flagship_final") if (BUNDLE_ROOT / "notebooks").exists() else (Path.cwd() / "outputs" / "flagship_final")
FLAGSHIP_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

from examples.default_settings import SETTINGS as BASE_SETTINGS

In [2]:
from flagship_predictability import run_truth_regimes

SETTINGS = deepcopy(BASE_SETTINGS)
# Main paper defaults
SETTINGS.n_regimes = 4
SETTINGS.n_eof_components = 8
SETTINGS.blocking_threshold = 0.1

display(Markdown("### Effective flagship settings"))
SETTINGS

### Effective flagship settings

AtlasConfig(truth_dataset='era5_truth_240', deterministic_models={'HRES': 'hres_0012_240', 'GraphCast': 'graphcast_2020_240', 'NeuralGCM': 'neuralgcm_det_2020_240'}, ensemble_models={'IFS-ENS': 'ifs_ens_240'}, date_windows=[('2020-01-01', '2020-03-31')], leads_hours=[24, 72, 120, 168], variables={'z500': VariableSpec(name='z500', candidates=['z', 'geopotential', 'gh'], level=500, climatology_groupby=None, thresholds=[]), 't850': VariableSpec(name='t850', candidates=['t', 'temperature'], level=850, climatology_groupby=None, thresholds=[]), 'u850': VariableSpec(name='u850', candidates=['u', 'u_component_of_wind'], level=850, climatology_groupby=None, thresholds=[]), 'v850': VariableSpec(name='v850', candidates=['v', 'v_component_of_wind'], level=850, climatology_groupby=None, thresholds=[]), 'mslp': VariableSpec(name='mslp', candidates=['msl', 'mean_sea_level_pressure', 'mslp'], level=None, climatology_groupby=None, thresholds=[])}, n_regimes=4, n_eof_components=8, bootstrap_n=400, boots

In [3]:
out = run_truth_regimes(SETTINGS, output_root=FLAGSHIP_OUTPUT_ROOT)
out

{'regime_labels_csv': PosixPath('/global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass/notebooks/outputs/flagship_final/regimes/regime_labels.csv')}

In [4]:
from pathlib import Path

for key, path in out.items():
    path = Path(path)
    print(f"\n--- {key} ---")
    print(path)
    if path.suffix.lower() == ".csv" and path.exists():
        try:
            display(pd.read_csv(path).head(10))
        except Exception as e:
            print("Could not preview CSV:", e)


--- regime_labels_csv ---
/global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass/notebooks/outputs/flagship_final/regimes/regime_labels.csv


,time,regime,window
0,2020-01-01 00:00:00,0,2020-01-01_2020-03-31
1,2020-01-01 06:00:00,0,2020-01-01_2020-03-31
2,2020-01-01 12:00:00,0,2020-01-01_2020-03-31
3,2020-01-01 18:00:00,0,2020-01-01_2020-03-31
4,2020-01-02 00:00:00,2,2020-01-01_2020-03-31
5,2020-01-02 06:00:00,2,2020-01-01_2020-03-31
6,2020-01-02 12:00:00,2,2020-01-01_2020-03-31
7,2020-01-02 18:00:00,2,2020-01-01_2020-03-31
8,2020-01-03 00:00:00,2,2020-01-01_2020-03-31
9,2020-01-03 06:00:00,2,2020-01-01_2020-03-31


In [5]:
labels = pd.read_csv(out["regime_labels_csv"])
count_df = labels["regime"].value_counts().sort_index().rename_axis("regime").reset_index(name="count")
count_df["fraction"] = count_df["count"] / count_df["count"].sum()
display(Markdown("### Regime balance"))
display(count_df)

status = "PASS" if (count_df["fraction"].max() < 0.6 and count_df["fraction"].min() > 0.08) else "WARN"
display(Markdown(f"**Regime-balance status:** {status}"))

### Regime balance

,regime,count,fraction
0,0,84,0.230769
1,1,43,0.118132
2,2,129,0.354396
3,3,108,0.296703


**Regime-balance status:** PASS